# Test netCDF format dtopo files

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools, dtopotools
import xarray as xr

In [ ]:
BL10M = xr.open_dataset('BL10M.nc')

In [ ]:
BL10M

In [ ]:
x = BL10M['lon'].to_numpy()
y = BL10M['lat'].to_numpy()
t = BL10M['t'].to_numpy()
dZ = BL10M['vert_disp'].to_numpy()
print(f'x has shape {x.shape} from {x[0]} to {x[-1]}')
print(f'y has shape {y.shape} from {y[0]} to {y[-1]}')
print(f't has shape {t.shape} from {t[0]} to {t[-1]}')
print(f'dZ has shape {dZ.shape}')


In [ ]:
print(f'Min dZ = {dZ.min():.2f},  Max dZ = {dZ.max():.2f}')

In [ ]:
coast = load('../CSZ_coast.npy')

In [ ]:
fig,ax = subplots(figsize=(8,10))
ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
ax.set_aspect(1/cos(45*pi/180))
ax.set_xlim(-130,-121)
ax.set_ylim(39,50)

clines = arange(-3,20,2)
ax.contour(x, y, dZ[-1,:,:], clines, colors='k')

## Make GeoClaw dtopo file

**NOTE:** It is necessary to flip the `lat` array `y` and the deformation array `dZ` since in the netCDF file `lat` is a decreasing array from 50N to 40N.

In [ ]:
dtopo = dtopotools.DTopography()
dtopo.x = x
dtopo.y = flipud(y)  # so increasing
dtopo.X, dtopo.Y = meshgrid(dtopo.x,dtopo.y)
dtopo.dZ = flip(dZ, axis=1)  # to match y
dtopo.times = t

fname = 'BL10M_dtopo.asc'
dtopo.write(fname, dtopo_type=3)
print('Created ', fname)

In [ ]:
print(f'dtopo.x has shape {dtopo.x.shape} from {dtopo.x[0]} to {dtopo.x[-1]}')
print(f'dtopo.y has shape {dtopo.y.shape} from {dtopo.y[0]} to {dtopo.y[-1]}')
print(f'dtopo.times has shape {dtopo.times.shape} from {dtopo.times[0]} to {dtopo.times[-1]}')
print(f'dtopo.dZ has shape {dtopo.dZ.shape}')

In [ ]:
fig,ax = subplots(figsize=(8,10))
ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
ax.set_aspect(1/cos(45*pi/180))
ax.set_xlim(-130,-121)
ax.set_ylim(39,50)

dtopo.plot_dZ_colors(t=t.max(), axes=ax, dZ_interval=4, cmax_dZ=10)

### Test by reloading and plotting

In [ ]:
fig,ax = subplots(figsize=(8,10))
ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
ax.set_aspect(1/cos(45*pi/180))
ax.set_xlim(-130,-121)
ax.set_ylim(39,50)

dtopo2 = dtopotools.DTopography(fname, dtopo_type=3)
dtopo2.plot_dZ_colors(t=t.max(), axes=ax, dZ_interval=4, cmax_dZ=10)

## Try changing data format from `float64` to `float32`

This should be ample resolution (6 digits) and *should* give netCDF files that are half the size.

In [ ]:
file_encoding = {"lat": {"dtype": "float32"},
                 "lon": {"dtype": "float32"},
                 "t": {"dtype": "float32"},
                 "vert_disp": {"dtype": "float32"},
                }

# write out using xarray:
fname = 'BL10M_float32.nc'
BL10M.to_netcdf(fname, encoding=file_encoding)

But for some reason this results in a larger file (126M rather than 55M):

In [ ]:
ls -lh BL10M*nc

### Read back in to check:

In [ ]:
BL10M_float32 = xr.open_dataset(fname)

In [ ]:
BL10M_float32

In [ ]:
dZ_float32 = BL10M_float32['vert_disp'].to_numpy()

In [ ]:
dZ_float32.dtype

In [ ]:
print(f'Max difference in dZ is {abs(dZ - dZ_float32).max():.10f} meters')